In [1]:
# ============================================================
# SHAMPOO SALES ANALYSIS USING NLP + K-MEANS CLUSTERING
# ============================================================

# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")


# ============================================================
# CELL 2: DOWNLOAD NLTK DATA
# ============================================================

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

print("NLTK resources downloaded successfully!")


# ============================================================
# CELL 3: LOAD DATASET
# ============================================================

file_path = "sales-of-shampoo.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape of dataset:", df.shape)

display(df.head())


# ============================================================
# CELL 4: CHECK DATASET INFORMATION
# ============================================================

print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


# ============================================================
# CELL 5: CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

print("Cleaned column names:")
print(df.columns.tolist())


# ============================================================
# CELL 6: CLEAN SALES DATA
# ============================================================

# Convert Sales column to numeric
df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")

# Remove rows with missing values
df = df.dropna(subset=["Month", "Sales"])

# Reset index
df = df.reset_index(drop=True)

print("Dataset after cleaning:")
display(df.head())

print("\nShape after cleaning:", df.shape)


# ============================================================
# CELL 7: BASIC STATISTICAL ANALYSIS
# ============================================================

print("Descriptive statistics:")
display(df.describe())


# ============================================================
# CELL 8: NLP - CONVERT MONTH TO TEXT
# ============================================================

# Convert Month column into string
df["Month_Text"] = df["Month"].astype(str)

# Convert text to lowercase
df["Month_Text"] = df["Month_Text"].str.lower()

display(df[["Month", "Month_Text"]].head())


# ============================================================
# CELL 9: NLP - TOKENIZATION
# ============================================================

df["Tokens"] = df["Month_Text"].apply(word_tokenize)

display(df[["Month", "Month_Text", "Tokens"]].head())


# ============================================================
# CELL 10: NLP - REMOVE STOPWORDS
# ============================================================

stop_words = set(stopwords.words("english"))

def remove_stopwords(tokens):
    cleaned_tokens = []

    for word in tokens:
        if word.isalpha() and word not in stop_words:
            cleaned_tokens.append(word)

    return cleaned_tokens


df["Clean_Tokens"] = df["Tokens"].apply(remove_stopwords)

display(
    df[
        [
            "Month",
            "Month_Text",
            "Tokens",
            "Clean_Tokens"
        ]
    ].head()
)


# ============================================================
# CELL 11: EXTRACT MONTH AND YEAR
# ============================================================

# The dataset normally contains dates such as:
# 1-01, 1-02, 1-03, etc.

df["Month_Name"] = df["Month_Text"].str.split("-").str[-1]

display(
    df[
        [
            "Month",
            "Month_Name",
            "Sales"
        ]
    ].head(10)
)


# ============================================================
# CELL 12: CREATE TIME INDEX
# ============================================================

# Create a numerical time index for clustering

df["Time_Index"] = np.arange(len(df))

display(
    df[
        [
            "Month",
            "Time_Index",
            "Sales"
        ]
    ].head(10)
)


# ============================================================
# CELL 13: SALES VISUALIZATION
# ============================================================

plt.figure(figsize=(14, 6))

plt.plot(
    df["Time_Index"],
    df["Sales"],
    marker="o"
)

plt.title("Shampoo Sales Over Time")
plt.xlabel("Time Index")
plt.ylabel("Sales")
plt.grid(True)

plt.show()


# ============================================================
# CELL 14: SALES HISTOGRAM
# ============================================================

plt.figure(figsize=(10, 6))

plt.hist(
    df["Sales"],
    bins=10
)

plt.title("Distribution of Shampoo Sales")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.grid(True)

plt.show()


# ============================================================
# CELL 15: PREPARE DATA FOR K-MEANS
# ============================================================

# We use:
# 1. Time_Index
# 2. Sales

X = df[
    [
        "Time_Index",
        "Sales"
    ]
]

print("Features used for K-Means:")
display(X.head())


# ============================================================
# CELL 16: STANDARDIZE DATA
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Scaled data:")
print(X_scaled[:10])


# ============================================================
# CELL 17: ELBOW METHOD
# ============================================================

inertia = []

K = range(1, 11)

for k in K:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)

    inertia.append(kmeans.inertia_)


# Plot elbow curve

plt.figure(figsize=(10, 6))

plt.plot(
    K,
    inertia,
    marker="o"
)

plt.title("Elbow Method for Finding Optimal K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.xticks(K)
plt.grid(True)

plt.show()


# ============================================================
# CELL 18: SILHOUETTE ANALYSIS
# ============================================================

silhouette_scores = []

for k in range(2, 11):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_scaled)

    score = silhouette_score(
        X_scaled,
        labels
    )

    silhouette_scores.append(score)


# Display scores

silhouette_df = pd.DataFrame({
    "Number_of_Clusters": range(2, 11),
    "Silhouette_Score": silhouette_scores
})

display(silhouette_df)


# ============================================================
# CELL 19: PLOT SILHOUETTE SCORES
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    silhouette_df["Number_of_Clusters"],
    silhouette_df["Silhouette_Score"],
    marker="o"
)

plt.title("Silhouette Score Analysis")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.grid(True)

plt.show()


# ============================================================
# CELL 20: SELECT NUMBER OF CLUSTERS
# ============================================================

# Here we select 3 clusters.
# You can change this value based on the Elbow Method
# and Silhouette Score.

optimal_k = 3

print("Selected number of clusters:", optimal_k)


# ============================================================
# CELL 21: APPLY K-MEANS
# ============================================================

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(X_scaled)

print("K-Means clustering completed!")

display(
    df[
        [
            "Month",
            "Sales",
            "Cluster"
        ]
    ].head(20)
)


# ============================================================
# CELL 22: CLUSTER CENTERS
# ============================================================

cluster_centers_scaled = kmeans.cluster_centers_

# Convert cluster centers back to original scale
cluster_centers = scaler.inverse_transform(
    cluster_centers_scaled
)

cluster_centers_df = pd.DataFrame(
    cluster_centers,
    columns=[
        "Time_Index",
        "Sales"
    ]
)

cluster_centers_df["Cluster"] = range(optimal_k)

print("Cluster Centers:")
display(cluster_centers_df)


# ============================================================
# CELL 23: VISUALIZE K-MEANS CLUSTERS
# ============================================================

plt.figure(figsize=(14, 7))

for cluster in sorted(df["Cluster"].unique()):

    cluster_data = df[
        df["Cluster"] == cluster
    ]

    plt.scatter(
        cluster_data["Time_Index"],
        cluster_data["Sales"],
        s=100,
        label=f"Cluster {cluster}"
    )


# Plot cluster centers

plt.scatter(
    cluster_centers_df["Time_Index"],
    cluster_centers_df["Sales"],
    marker="X",
    s=250,
    label="Cluster Centers"
)

plt.title("K-Means Clustering of Shampoo Sales")
plt.xlabel("Time Index")
plt.ylabel("Sales")
plt.legend()
plt.grid(True)

plt.show()


# ============================================================
# CELL 24: SILHOUETTE SCORE FOR FINAL MODEL
# ============================================================

final_silhouette_score = silhouette_score(
    X_scaled,
    df["Cluster"]
)

print(
    "Final Silhouette Score:",
    round(final_silhouette_score, 4)
)


# ============================================================
# CELL 25: CLUSTER SUMMARY
# ============================================================

cluster_summary = df.groupby(
    "Cluster"
)["Sales"].agg(
    [
        "count",
        "mean",
        "median",
        "min",
        "max",
        "sum"
    ]
)

print("Cluster Summary:")
display(cluster_summary)


# ============================================================
# CELL 26: SORT CLUSTERS BY AVERAGE SALES
# ============================================================

cluster_average_sales = (
    df.groupby("Cluster")["Sales"]
    .mean()
    .sort_values()
)

print("Average sales by cluster:")
display(cluster_average_sales)


# ============================================================
# CELL 27: FIND LOW, MEDIUM AND HIGH SALES CLUSTERS
# ============================================================

sorted_clusters = cluster_average_sales.index.tolist()

low_sales_cluster = sorted_clusters[0]

if len(sorted_clusters) > 2:
    medium_sales_cluster = sorted_clusters[1]
else:
    medium_sales_cluster = None

high_sales_cluster = sorted_clusters[-1]

print("Low-sales cluster:", low_sales_cluster)

if medium_sales_cluster is not None:
    print("Medium-sales cluster:", medium_sales_cluster)

print("High-sales cluster:", high_sales_cluster)


# ============================================================
# CELL 28: DISPLAY LOW SALES PERIODS
# ============================================================

print("Low Sales Periods:")

display(
    df[
        df["Cluster"] == low_sales_cluster
    ][
        [
            "Month",
            "Sales",
            "Cluster"
        ]
    ].sort_values("Sales")
)


# ============================================================
# CELL 29: DISPLAY HIGH SALES PERIODS
# ============================================================

print("High Sales Periods:")

display(
    df[
        df["Cluster"] == high_sales_cluster
    ][
        [
            "Month",
            "Sales",
            "Cluster"
        ]
    ].sort_values(
        "Sales",
        ascending=False
    )
)


# ============================================================
# CELL 30: CLUSTER DISTRIBUTION
# ============================================================

cluster_counts = df["Cluster"].value_counts().sort_index()

print("Number of records in each cluster:")
display(cluster_counts)


# ============================================================
# CELL 31: CLUSTER DISTRIBUTION BAR CHART
# ============================================================

plt.figure(figsize=(8, 5))

plt.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

plt.title("Number of Months in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Months")
plt.grid(axis="y")

plt.show()


# ============================================================
# CELL 32: MONTHLY SALES WITH CLUSTERS
# ============================================================

plt.figure(figsize=(15, 7))

plt.plot(
    df["Time_Index"],
    df["Sales"],
    marker="o"
)

for i in range(len(df)):

    plt.annotate(
        str(df.loc[i, "Cluster"]),
        (
            df.loc[i, "Time_Index"],
            df.loc[i, "Sales"]
        ),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center"
    )

plt.title("Shampoo Sales with K-Means Cluster Labels")
plt.xlabel("Time Index")
plt.ylabel("Sales")
plt.grid(True)

plt.show()


# ============================================================
# CELL 33: CORRELATION
# ============================================================

correlation = df[
    [
        "Time_Index",
        "Sales"
    ]
].corr()

print("Correlation Matrix:")
display(correlation)


# ============================================================
# CELL 34: SALES GROWTH
# ============================================================

df["Sales_Change"] = df["Sales"].diff()

df["Sales_Growth_Percentage"] = (
    df["Sales"].pct_change() * 100
)

display(
    df[
        [
            "Month",
            "Sales",
            "Sales_Change",
            "Sales_Growth_Percentage"
        ]
    ].head(15)
)


# ============================================================
# CELL 35: HIGHEST SALES
# ============================================================

highest_sales = df.loc[
    df["Sales"].idxmax()
]

print("Highest Sales:")
display(
    highest_sales[
        [
            "Month",
            "Sales",
            "Cluster"
        ]
    ]
)


# ============================================================
# CELL 36: LOWEST SALES
# ============================================================

lowest_sales = df.loc[
    df["Sales"].idxmin()
]

print("Lowest Sales:")
display(
    lowest_sales[
        [
            "Month",
            "Sales",
            "Cluster"
        ]
    ]
)


# ============================================================
# CELL 37: AVERAGE SALES
# ============================================================

average_sales = df["Sales"].mean()

print(
    "Average Shampoo Sales:",
    round(average_sales, 2)
)


# ============================================================
# CELL 38: TOTAL SALES
# ============================================================

total_sales = df["Sales"].sum()

print(
    "Total Shampoo Sales:",
    round(total_sales, 2)
)


# ============================================================
# CELL 39: FINAL DATASET
# ============================================================

print("Final Dataset:")

display(
    df[
        [
            "Month",
            "Sales",
            "Month_Text",
            "Tokens",
            "Clean_Tokens",
            "Time_Index",
            "Cluster"
        ]
    ]
)


# ============================================================
# CELL 40: SAVE FINAL DATASET
# ============================================================

output_file = "shampoo_sales_clustered.csv"

df.to_csv(
    output_file,
    index=False
)

print(
    "Final dataset saved successfully as:",
    output_file
)


# ============================================================
# CELL 41: FINAL RESULTS
# ============================================================

print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print("Number of records:", len(df))
print("Total sales:", round(total_sales, 2))
print("Average sales:", round(average_sales, 2))
print("Highest sales:", round(df["Sales"].max(), 2))
print("Lowest sales:", round(df["Sales"].min(), 2))
print("Number of clusters:", optimal_k)
print(
    "Silhouette Score:",
    round(final_silhouette_score, 4)
)

print("=" * 60)

print("\nCluster Average Sales:")
display(cluster_average_sales)

print("\nCluster Summary:")
display(cluster_summary)

Libraries imported successfully!
NLTK resources downloaded successfully!
Dataset loaded successfully!
Shape of dataset: (36, 2)


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Month,Sales of shampoo
0,1-01,266.0
1,1-02,145.9
2,1-03,183.1
3,1-04,119.3
4,1-05,180.3


Column names:
['Month', 'Sales of shampoo']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Month             36 non-null     str    
 1   Sales of shampoo  36 non-null     float64
dtypes: float64(1), str(1)
memory usage: 852.0 bytes

Missing values:
Month               0
Sales of shampoo    0
dtype: int64

Duplicate rows:
0
Cleaned column names:
['Month', 'Sales of shampoo']


KeyError: 'Sales'